# Lesson 5: Movement & Face Detection

## Task 1: Pedestrian Detection
1. Create a loop which can play through the video linked above. Add appropriate control
functionality using cv.WaitKey().
2. Create a background subtraction object outside of the loop using the following code and
function below:
background_sub = cv.createBackgroundSubtractorMOG2()
This object can be used to create a mask of a frame which detects objects moving against a
background with the following method call taking a frame as argument:
bg_mask = background_sub.apply(frame)
3. Display the background mask using cv.imshow() to inspect its output. What do you notice?
How is the output useful for detecting the people? Is any data cleaning/filtering required?
Hint: Think about applying a threshold.
4. A human is a shape fully enclosed by edges (in other words a contour). Detect all the
contours in the image.
5. Draw bounding boxes around the contours using an appropriate contour helper function.
Display the video with bounding boxes drawn over detected people, you may need to
resize the output video depending on the size of your computer monitor.
6. What do you notice about the output? Is it noisy and if so, what kind of noise is there?
What can be done to further filter out the noise? Hint: Check lesson 4 under contours.
7. Display your result, varying the level of noise filtering applied.

In [ ]:
import cv2 as cv
import numpy as np

def rescale(frame, scale):
    height = int(frame.shape[0] * scale)
    width = int(frame.shape[1] * scale)
    dim = (width, height)
    return cv.resize(frame, dim, interpolation=cv.INTER_AREA)

# Open the video file
capture = cv.VideoCapture("FASTR_CrowdedMall.mp4")

# Create background subtractor object BEFORE the loop
# MOG2 learns the static background over time and flags moving pixels
background_sub = cv.createBackgroundSubtractorMOG2()

# Minimum contour area — anything smaller is treated as noise 
# Increase = Larger moving objects, smaller = smaller groups but more noise
MIN_AREA = 400

while True:
    retval, frame = capture.read()

    if retval == False:
        break

    # Resize frame so it fits on screen
    frame = rescale(frame, 0.5)

    # Apply background subtraction to get a mask of moving objects
    # White pixels = movement, black pixels = static background
    bg_mask = background_sub.apply(frame)

    # Apply a threshold to clean up the mask
    # Pixels above 200 become white (255), everything else becomes black (0)
    _, thresh = cv.threshold(bg_mask, 200, 255, cv.THRESH_BINARY)

    # Apply morphological closing to fill gaps in detected shapes
    kernel = cv.getStructuringElement(cv.MORPH_RECT, (5, 5))
    cleaned = cv.morphologyEx(thresh, cv.MORPH_CLOSE, kernel, iterations=2)

    # Apply morphological opening to remove small noise spots
    cleaned = cv.morphologyEx(cleaned, cv.MORPH_OPEN, kernel, iterations=1)

    # Find contours in the cleaned mask
    contours, _ = cv.findContours(cleaned, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)

    # Make a copy to draw on
    result = frame.copy()

    # Loop through contours and draw bounding boxes
    for contour in contours:
        # Skip small contours — likely noise, not a person
        if cv.contourArea(contour) < MIN_AREA:
            continue

        # Get bounding rectangle around the contour
        x, y, w, h = cv.boundingRect(contour)

        # Draw a green bounding box on the result frame
        cv.rectangle(result, (x, y), (x + w, y + h), (0, 255, 0), 2)

    # Display all stages so you can see what each step does
    cv.imshow("Original", frame)
    cv.imshow("Background Mask", bg_mask)
    cv.imshow("Cleaned Mask", cleaned)
    cv.imshow("Detections", result)

    if cv.waitKey(1) & 0xFF == ord('d'):
        break

capture.release()
cv.destroyAllWindows()

## Task 2: Face Detection
Congratulations, you’ve reached the pinnacle of USRC’s OpenCV workshops. This is where you get to prove to us you can do this all by yourself (i.e. stfu i’m out). Your job is to create a face detector.

In [ ]:
import cv2 as cv
import numpy as np

def rescale(frame, scale):
    height = int(frame.shape[0] * scale)
    width = int(frame.shape[1] * scale)
    dim = (width, height)
    return cv.resize(frame, dim, interpolation=cv.INTER_AREA)

# Load the pretrained Haar Cascade classifier for face detection
# This XML file contains the trained model data
face_cascade = cv.CascadeClassifier(cv.data.haarcascades + "haarcascade_frontalface_default.xml")

# Open webcam (0) or video file
capture = cv.VideoCapture(0)

while True:
    retval, frame = capture.read()

    if retval == False:
        break

    # Resize frame to fit on screen
    frame = rescale(frame, 0.5)

    # Convert to grayscale — Haar Cascade requires grayscale input
    gray = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)

    # Detect faces in the grayscale frame
    # scaleFactor — how much the image is shrunk at each scale (1.1 = 10% reduction)
    # minNeighbors — how many detections needed to confirm a face (higher = fewer false positives)
    # minSize — smallest face size to detect in pixels
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))

    # Draw a bounding box and label around each detected face
    for (x, y, w, h) in faces:
        # Green rectangle around the face
        cv.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

        # Label above the bounding box
        cv.putText(frame, "Face", (x, y - 10), cv.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    # Show how many faces detected in the corner
    cv.putText(frame, f"Faces: {len(faces)}", (10, 30), cv.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

    # Display the result
    cv.imshow("Face Detection", frame)

    if cv.waitKey(1) & 0xFF == ord('d'):
        break

capture.release()
cv.destroyAllWindows()